In [7]:
# [Cell 1] 경로 지정: 원본/리라이트 JSON과 결과 저장 경로
dataname = "dev"
INPUT_ORIG_JSON = f"../dataset/original_formatted/{dataname}.json"   # 원본(리라이트 전) JSON 경로
INPUT_REW_JSON  = f"../dataset/naturalized/original_{dataname}.json"  # 리라이트 후 JSON 경로
OUTPUT_JSON     = f"../dataset/ut_finetuning_dataset/{dataname}.json"  # 저장 경로

### 유틸 함수 및 변환 파이프라인

In [ ]:
import json
import os
from typing import List, Tuple, Dict, Set

def parse_dialogue(text: str) -> List[Tuple[str, str]]:
    """대화 문자열 → [(speaker, utterance), ...]"""
    lines = []
    for raw in text.splitlines():
        s = raw.strip()
        if not s:
            continue
        if ":" in s:
            spk, utt = s.split(":", 1)
            lines.append((spk.strip(), utt.strip()))
    return lines

def join_dialogue(lines: List[Tuple[str, str]]) -> str:
    """[(speaker, utterance)] → '화자X: 내용' 줄 결합"""
    return "\n".join(f"{spk}: {utt}" for spk, utt in lines)

def build_samples_one_pair(
    orig_item: Dict, rew_item: Dict
) -> List[Dict]:
    """
    같은 id의 원본/리라이트 항목에서 학습 샘플 생성.
    dialogue = (지금까지의 모든 '리라이트 후' 발화들 [:i]) + (현재 i번째 '리라이트 전' 1줄)
    answer  = 현재 i번째 '리라이트 후' 발화 텍스트
    """
    id_ = orig_item.get("id")
    orig_pairs = parse_dialogue(orig_item.get("dialogue", ""))
    rew_pairs  = parse_dialogue(rew_item.get("dialogue", ""))

    n = min(len(orig_pairs), len(rew_pairs))
    if len(orig_pairs) != len(rew_pairs):
        print(f"[경고] id={id_} 원본({len(orig_pairs)})/리라이트({len(rew_pairs)}) 발화 수 불일치 → {n}만 사용")

    out = []
    for i in range(n):  # 0..n-1
        # 직전까지의 모든 리라이트 후 컨텍스트
        context_rew = rew_pairs[:i]              # ★ 핵심 변경: 전체 과거 컨텍스트
        curr_orig_pair = orig_pairs[i]           # 현재 발화(리라이트 전)
        merged_lines = context_rew + [curr_orig_pair]

        out.append({
            "id": id_,
            "utterance_num": i + 1,              # 1-based
            "dialogue": join_dialogue(merged_lines),
            "answer": rew_pairs[i][1],           # 현재 i의 리라이트 후 텍스트
        })
    return out

def collect_consecutive_duplicate_ids(items: List[Dict]) -> Set[str]:
    """
    리스트 순서 기준으로, '직전 항목과 dialogue 문자열이 완전히 동일'하면
    두 번째 항목의 id를 스킵 대상으로 수집.
    """
    skip_ids = set()
    prev_dialogue = None
    for it in items:
        d = (it.get("dialogue") or "").strip()
        if prev_dialogue is not None and d == prev_dialogue:
            if "id" in it:
                skip_ids.add(it["id"])
        prev_dialogue = d
    return skip_ids

# --- 입출력 ---
assert os.path.exists(INPUT_ORIG_JSON), f"원본 JSON이 없습니다: {INPUT_ORIG_JSON}"
assert os.path.exists(INPUT_REW_JSON),  f"리라이트 JSON이 없습니다: {INPUT_REW_JSON}"

with open(INPUT_ORIG_JSON, "r", encoding="utf-8") as f:
    orig_data = json.load(f)
with open(INPUT_REW_JSON, "r", encoding="utf-8") as f:
    rew_data = json.load(f)

# 1) 연속 중복 스킵 id 수집 (원본/리라이트 각각 검사)
skip_ids = set()
skip_ids |= collect_consecutive_duplicate_ids(orig_data)
skip_ids |= collect_consecutive_duplicate_ids(rew_data)

if skip_ids:
    print(f"[정보] 연속 중복으로 스킵할 id 수: {len(skip_ids)} (예시: {list(skip_ids)[:5]})")

# 2) 리라이트 데이터 매핑 (스킵 id 제외)
rew_by_id = {item["id"]: item for item in rew_data if item.get("id") not in skip_ids}

# 3) 매칭/생성
merged = []
missing, skipped = [], 0
for item in orig_data:
    id_ = item.get("id")
    if id_ in skip_ids:
        skipped += 1
        continue
    if id_ not in rew_by_id:
        missing.append(id_)
        continue
    merged.extend(build_samples_one_pair(item, rew_by_id[id_]))

if missing:
    print(f"[주의] 리라이트 세트에 없어 스킵한 id 수: {len(missing)} (예: {missing[:3]})")
if skipped:
    print(f"[완료] 연속 중복 감지로 스킵한 원본 id 수: {skipped}")

# 4) 저장
os.makedirs(os.path.dirname(OUTPUT_JSON) or ".", exist_ok=True)
with open(OUTPUT_JSON, "w", encoding="utf-8") as f:
    json.dump(merged, f, ensure_ascii=False, indent=2)

print(f"완료: {len(merged)}개 샘플 저장 → {OUTPUT_JSON}")

완료: 2360개 샘플 저장 → ../dataset/ut_finetuning_dataset/dev.json


### 빠른 검증: 몇 개 예시 출력

In [9]:
import itertools
from random import randint

def preview_samples(samples, k=3):
    take = list(itertools.islice(samples, k))
    for i, s in enumerate(take, 1):
        print(f"\n=== 예시 {i} ===")
        print(f"id: {s['id']}")
        print(f"utterance_num: {s['utterance_num']}")
        print("\n[dialogue]\n" + s["dialogue"])
        print("\n[answer]\n" + s["answer"])

with open(OUTPUT_JSON, "r", encoding="utf-8") as f:
    samples = json.load(f)

preview_samples(samples, k=3)


=== 예시 1 ===
id: nikluge-2025-일상 대화 요약-dev-000001
utterance_num: 1

[dialogue]
화자1: 근데 인제 건강이 우리만 막 건강한다고 되는 게 아니잖아. 인제 애들도 건강도 챙겨줘야 되고 그러잖아. 그러니까 너는 애들한테 뭐 따로 먹이는 뭐 식품 뭐 이런 거 있어?

[answer]
그런데 이제 건강은 우리만 챙긴다고 되는 게 아니잖아. 아이들도 건강을 챙겨줘야 하니까, 너는 아이들에게 따로 먹이는 식품 같은 게 있어?

=== 예시 2 ===
id: nikluge-2025-일상 대화 요약-dev-000001
utterance_num: 2

[dialogue]
화자1: 그런데 이제 건강은 우리만 챙긴다고 되는 게 아니잖아. 아이들도 건강을 챙겨줘야 하니까, 너는 아이들에게 따로 먹이는 식품 같은 게 있어?
화자2: 유산 유산균하고 비타민 그런데 애들이 잘 안 먹지. 나도 잘 안 먹는 데 애들이 먹나? 근데 쫓아다니면서 챙겨줄 수도 없고 그런데 유산균은 꼭 먹이라 그러더라고. 유산균 장이 건강해야지 전체적으로 다 좋아진다고 그러더라고.

[answer]
유산균과 비타민을 주긴 하는데, 애들이 잘 안 먹더라고요. 저도 잘 안 먹는데 애들이 먹겠어요? 그런데 쫓아다니면서 챙겨줄 수도 없고, 유산균은 꼭 먹이라고 하더라고요. 유산균이 장 건강에 좋으면 전체적으로 다 좋아진다고 하니까요.

=== 예시 3 ===
id: nikluge-2025-일상 대화 요약-dev-000001
utterance_num: 3

[dialogue]
화자1: 그런데 이제 건강은 우리만 챙긴다고 되는 게 아니잖아. 아이들도 건강을 챙겨줘야 하니까, 너는 아이들에게 따로 먹이는 식품 같은 게 있어?
화자2: 유산균과 비타민을 주긴 하는데, 애들이 잘 안 먹더라고요. 저도 잘 안 먹는데 애들이 먹겠어요? 그런데 쫓아다니면서 챙겨줄 수도 없고, 유산균은 꼭 먹이라고 하더라고요. 유산균이 장 건강에 좋으면 전체적으로 다 좋아진다고 하니까요.
